# LangGraph with AgentCore Memory Hooks (Long-term Memory)

# LangGraph 与 AgentCore Memory 钩子（长期记忆）

## Introduction

## 简介

This notebook demonstrates how to integrate Amazon Bedrock AgentCore Memory capabilities with a conversational AI agent using LangGraph framework. We'll focus on **long-term memory** retention across multiple conversation sessions - allowing an agent to extract and recall user preferences, dietary restrictions, and contextual information from past interactions.

本笔记本演示如何使用 LangGraph 框架将 Amazon Bedrock AgentCore Memory 功能与对话式 AI 代理集成。我们将专注于跨多个对话会话的**长期记忆**保留 - 允许代理从过去的交互中提取和回忆用户偏好、饮食限制和上下文信息。

## Tutorial Details

## 教程详情

| Information         | Details                                                                          |
|:--------------------|:---------------------------------------------------------------------------------|
| Tutorial type       | Long-term Conversational                                                        |
| Agent usecase       | Nutrition Assistant                                                              |
| Agentic Framework   | LangGraph                                                                        |
| LLM model           | Anthropic Claude Haiku 4.5                                                     |
| Tutorial components | AgentCore Long-term Memory, Custom Memory Strategies, Pre/Post Model Hooks     |
| Example complexity  | Intermediate                                                                     |

| 信息                 | 详情                                                                              |
|:--------------------|:---------------------------------------------------------------------------------|
| 教程类型             | 长期对话                                                                          |
| 代理用例             | 营养助手                                                                          |
| 代理框架             | LangGraph                                                                        |
| LLM 模型             | Anthropic Claude Haiku 4.5                                                      |
| 教程组件             | AgentCore 长期记忆、自定义记忆策略、模型前/后钩子                                    |
| 示例复杂度           | 中级                                                                              |

You'll learn to:

您将学习：

- Create AgentCore Memory with UserPreference custom-override strategy
- Implement pre/post model hooks for automatic memory storage and retrieval
- Build a nutrition assistant that remembers user preferences across sessions
- Use semantic search to retrieve relevant user context
- Configure custom memory extraction and consolidation prompts

- 使用 UserPreference 自定义覆盖策略创建 AgentCore Memory
- 实现用于自动记忆存储和检索的模型前/后钩子
- 构建一个跨会话记住用户偏好的营养助手
- 使用语义搜索检索相关的用户上下文
- 配置自定义记忆提取和整合提示

### Scenario Context

### 场景背景

In this example, we'll create a **Nutrition Assistant** that can remember user context across multiple conversations, including dietary restrictions, favorite foods, cooking preferences, and health goals. The agent will automatically extract and store user preferences from conversations, then retrieve relevant context for future interactions to provide personalized nutrition advice.

在本示例中，我们将创建一个**营养助手**，它可以跨多个对话记住用户上下文，包括饮食限制、喜欢的食物、烹饪偏好和健康目标。代理将自动从对话中提取和存储用户偏好，然后为未来的交互检索相关上下文以提供个性化的营养建议。

## Architecture

## 架构

<div style="text-align:left">
    <img src="architecture.png" width="65%" />
</div>

## Prerequisites

## 前提条件

- Python 3.10+
- AWS account with appropriate permissions
- AWS IAM role with appropriate permissions for AgentCore Memory
- Access to Amazon Bedrock models

- Python 3.10+
- 具有适当权限的 AWS 账户
- 具有 AgentCore Memory 适当权限的 AWS IAM 角色
- 访问 Amazon Bedrock 模型

Let's get started by setting up our environment!

让我们开始设置环境吧！

In [ ]:
# Install necessary libraries from https://github.com/langchain-ai/langchain-aws
%pip install -qr requirements.txt

In [ ]:
import os
import logging

# Import LangGraph and LangChain components
from langchain.chat_models import init_chat_model
from langgraph.prebuilt import create_react_agent
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.runnables import RunnableConfig
from langgraph.store.base import BaseStore
import uuid


region = os.getenv("AWS_REGION", "us-east-1")
logging.getLogger("math-agent").setLevel(logging.DEBUG)

In [ ]:
# Import the AgentCoreMemoryStore that we will use as a store
from langgraph_checkpoint_aws import AgentCoreMemoryStore

# For this example, we will just use an InMemorySaver to save context.
# In production, we highly recommend the AgentCoreMemorySaver as a checkpointer which works seamlessly alongside the memory store
# from langgraph_checkpoint_aws import AgentCoreMemorySaver
from langgraph.checkpoint.memory import InMemorySaver
from bedrock_agentcore.memory import MemoryClient
from bedrock_agentcore.memory.constants import StrategyType

from custom_memory_prompts import consolidation_prompt, extraction_prompt

In [ ]:
memory_name = "NutritionAssistant"
client = MemoryClient(region_name=region)
MODEL_ID = "global.anthropic.claude-haiku-4-5-20251001-v1:0"

memory = client.create_or_get_memory(
    name=memory_name,
    description="Nutrition assistant",
    memory_execution_role_arn="arn:aws:iam::YOUR_ACCOUNT:role/YOUR_ROLE",  # Please provide a role with a valid trust policy
    strategies=[
        {
            StrategyType.CUSTOM.value: {
                "name": "NutritionPreferences",
                "description": "Captures customer food preferences and behavior",
                "namespaces": ["/{actorId}/preferences"],
                "configuration": {
                    "userPreferenceOverride": {
                        "extraction": {
                            "appendToPrompt": extraction_prompt,
                            "modelId": MODEL_ID,
                        },
                        "consolidation": {
                            "appendToPrompt": consolidation_prompt,
                            "modelId": MODEL_ID,
                        },
                    }
                },
            }
        },
    ],
)
memory_id = memory["id"]

### Memory Configuration Overview

### 记忆配置概述

Our AgentCore Memory setup includes:

我们的 AgentCore Memory 设置包括：

- **Custom Strategy**: Extracts nutrition preferences from conversations
- **Namespaces**: Organizes memories by user (`{actorId}/preferences`)
- **Custom Prompts**: Specialized extraction and consolidation logic for food preferences
- **Model Integration**: Uses Claude 3.7 Sonnet for memory processing

- **自定义策略**：从对话中提取营养偏好
- **命名空间**：按用户组织记忆（`{actorId}/preferences`）
- **自定义提示**：用于食物偏好的专业提取和整合逻辑
- **模型集成**：使用 Claude 3.7 Sonnet 进行记忆处理

The memory system will automatically process conversations to extract lasting user preferences while filtering out temporary or irrelevant information.

记忆系统将自动处理对话以提取持久的用户偏好，同时过滤掉临时或不相关的信息。

## Step 3: Initialize Memory Store and LLM

## 第三步：初始化记忆存储和 LLM

Now we'll initialize the AgentCore Memory Store and our language model.

现在我们将初始化 AgentCore Memory Store 和我们的语言模型。

In [ ]:
# Initialize the store to enable long term memory saving and retrieval
store = AgentCoreMemoryStore(memory_id=memory_id, region_name=region)

# Initialize Bedrock LLM
llm = init_chat_model(MODEL_ID, model_provider="bedrock_converse", region_name=region)

## Step 4: Implement Memory Hooks

## 第四步：实现记忆钩子

We'll create pre and post model hooks to automatically handle memory storage and retrieval:

我们将创建模型前和模型后钩子来自动处理记忆存储和检索：

- **Pre-model hook**: Retrieves relevant user preferences (based on semantic search) and adds context before LLM invocation
- **Post-model hook**: Saves the conversation messages for long-term memory extraction

- **模型前钩子**：检索相关的用户偏好（基于语义搜索）并在 LLM 调用前添加上下文
- **模型后钩子**：保存对话消息以进行长期记忆提取

### How Memory Processing Works

### 记忆处理工作原理

1. Messages are saved to AgentCore Memory with actor_id and session_id
2. The custom strategy processes conversations to extract nutrition preferences
3. Extracted preferences are stored in the `{actorId}/preferences` namespace
4. Future conversations can search and retrieve relevant preferences for context

1. 消息通过 actor_id 和 session_id 保存到 AgentCore Memory
2. 自定义策略处理对话以提取营养偏好
3. 提取的偏好存储在 `{actorId}/preferences` 命名空间中
4. 未来的对话可以搜索和检索相关偏好作为上下文

**Note**: LangChain message types are converted under the hood by the store to AgentCore Memory message types so that they can be properly extracted to long term memories.

**注意**：LangChain 消息类型在底层被存储转换为 AgentCore Memory 消息类型，以便可以正确提取到长期记忆中。

In [ ]:
def pre_model_hook(state, config: RunnableConfig, *, store: BaseStore):
    """Hook that runs pre-LLM invocation to save the latest human message"""
    actor_id = config["configurable"]["actor_id"]
    thread_id = config["configurable"]["thread_id"]
    # Saving the message to the actor and session combination that we get at runtime
    namespace = (actor_id, thread_id)

    messages = state.get("messages", [])
    # Save the last human message we see before LLM invocation
    for msg in reversed(messages):
        if isinstance(msg, HumanMessage):
            store.put(namespace, str(uuid.uuid4()), {"message": msg})
            break
    # Retrieve user preferences based on the last message and append to state
    user_preferences_namespace = (actor_id, "preferences")
    preferences = store.search(user_preferences_namespace, query=msg.content, limit=5)

    # Construct another AI message to add context before the current message
    if preferences:
        context_items = [pref.value for pref in preferences]
        context_message = AIMessage(
            content=f"[User Context: {', '.join(str(item) for item in context_items)}]"
        )
        # Insert the context message before the last human message
        return {"messages": messages[:-1] + [context_message, messages[-1]]}

    return {"llm_input_messages": messages}


def post_model_hook(state, config: RunnableConfig, *, store: BaseStore):
    """Hook that runs post-LLM invocation to save the latest human message"""
    actor_id = config["configurable"]["actor_id"]
    thread_id = config["configurable"]["thread_id"]

    # Saving the message to the actor and session combination that we get at runtime
    namespace = (actor_id, thread_id)

    messages = state.get("messages", [])
    # Save the LLMs response to AgentCore Memory
    for msg in reversed(messages):
        if isinstance(msg, AIMessage):
            store.put(namespace, str(uuid.uuid4()), {"message": msg})
            break

    return {"messages": messages}

## Step 5: Create the LangGraph Agent

## 第五步：创建 LangGraph 代理

Now we'll create our nutrition assistant agent using LangGraph's `create_react_agent` with our memory hooks integrated. The tool node will contain just our long term memory retrieval tool and the pre and post model hooks are specified as arguments.

现在我们将使用 LangGraph 的 `create_react_agent` 创建我们的营养助手代理，并集成我们的记忆钩子。工具节点将只包含我们的长期记忆检索工具，模型前和模型后钩子作为参数指定。

**Note**: for custom agent implementations the Store and tools can be configured to run as needed for any workflow following this pattern. Pre/post model hooks can be used, the whole conversation could be saved at the end, etc.

**注意**：对于自定义代理实现，Store 和工具可以按照此模式为任何工作流配置为按需运行。可以使用模型前/后钩子，也可以在最后保存整个对话等。

In [ ]:
graph = create_react_agent(
    llm,
    store=store,
    tools=[],  # No additional tools needed for this example
    checkpointer=InMemorySaver(),  # For conversation state management
    pre_model_hook=pre_model_hook,  # Retrieves user preferences before LLM call
    post_model_hook=post_model_hook,  # Saves conversation after LLM response
)

## Step 6: Configure Agent Runtime

## 第六步：配置代理运行时

We need to configure the agent with unique identifiers for the user and session. These IDs are crucial for memory organization and retrieval.

我们需要为用户和会话配置代理的唯一标识符。这些 ID 对于记忆组织和检索至关重要。

### Graph Invoke Input

### 图调用输入

We only need to pass the newest user message in as an argument `inputs`. This could include other state variables as well but for the simple `create_react_agent`, we only need messages.

我们只需要将最新的用户消息作为参数 `inputs` 传入。这也可以包含其他状态变量，但对于简单的 `create_react_agent`，我们只需要消息。

### LangGraph RuntimeConfig

### LangGraph 运行时配置

In LangGraph, config is a `RuntimeConfig` that contains attributes that are necessary at invocation time, for example user IDs or session IDs. For the `AgentCoreMemorySaver`, `thread_id` and `actor_id` must be set in the config. For instance, your AgentCore invocation endpoint could assign this based on the identity or user ID of the caller. You can read additional [documentation here](https://langchain-ai.github.io/langgraphjs/how-tos/configuration/)

在 LangGraph 中，config 是一个 `RuntimeConfig`，包含调用时必需的属性，例如用户 ID 或会话 ID。对于 `AgentCoreMemorySaver`，必须在配置中设置 `thread_id` 和 `actor_id`。例如，您的 AgentCore 调用端点可以根据调用者的身份或用户 ID 来分配这些值。您可以在[此处阅读更多文档](https://langchain-ai.github.io/langgraphjs/how-tos/configuration/)

In [ ]:
actor_id = "user-1"
config = {
    "configurable": {
        "thread_id": "session-1",  # REQUIRED: This maps to Bedrock AgentCore session_id under the hood
        "actor_id": actor_id,  # REQUIRED: This maps to Bedrock AgentCore actor_id under the hood
    }
}

## Step 7: Test the Agent

## 第七步：测试代理

Let's test our nutrition assistant by having a conversation about food preferences. The agent will automatically extract and store user preferences for future use.

让我们通过一次关于食物偏好的对话来测试我们的营养助手。代理将自动提取和存储用户偏好以供将来使用。

In [ ]:
# Helper function to pretty print agent output while running
def run_agent(query: str, config: RunnableConfig):
    printed_ids = set()
    events = graph.stream(
        {"messages": [{"role": "user", "content": query}]},
        config,
        stream_mode="values",
    )
    for event in events:
        if "messages" in event:
            for msg in event["messages"]:
                # Check if we've already printed this message
                if id(msg) not in printed_ids:
                    msg.pretty_print()
                    printed_ids.add(id(msg))


prompt = """
Hey there! Im cooking one of my favorite meals tonight, salmon with rice and veggies (healthy). Has
great macros for my weightlifting competition that is coming up. What can I add to this dish to make it taste better
and also improve the protein and vitamins I get?
"""

run_agent(prompt, config)

### What was stored?

### 存储了什么？

As you can see, the model does not yet have any insight into our preferences or dietary restrictions.

如您所见，模型尚未了解我们的偏好或饮食限制。

For this implementation with pre/post model hooks, two messages were stored here. The first message from the user and the response from the AI model were both stored as conversational events in AgentCore Memory. It may take a few moments for the long term memories to be extracted, so retry after a few seconds if nothing is found the first try.

对于这个使用模型前/后钩子的实现，这里存储了两条消息。来自用户的第一条消息和来自 AI 模型的响应都作为对话事件存储在 AgentCore Memory 中。长期记忆的提取可能需要一些时间，所以如果第一次没有找到任何内容，请几秒钟后重试。

These messages were then extracted to AgentCore long term memory in our fact and user preferences namespaces. In fact, we can check the store ourselves to verify what has been stored there so far:

这些消息随后被提取到我们的事实和用户偏好命名空间中的 AgentCore 长期记忆中。事实上，我们可以自己检查存储来验证到目前为止存储了什么：

In [ ]:
# Search our user preferences namespace
search_namespace = (actor_id, "preferences")
result = store.search(search_namespace, query="food", limit=3)
print(f"Preferences namespace result: {result}")

### Agent access to the store

### 代理访问存储

**Note** - since AgentCore memory processes these events in the background, it may take a few seconds for the memory to be extracted and embedded to long term memory retrieval.

**注意** - 由于 AgentCore memory 在后台处理这些事件，记忆的提取和嵌入到长期记忆检索可能需要几秒钟。

Great! Now we have seen that long term memories were extracted to our namespaces based on the earlier messages in the conversation.

很好！现在我们已经看到长期记忆根据对话中的早期消息被提取到我们的命名空间中。

Now, let's start a new session and ask about recommendations for what to cook for dinner. The agent can use the store to access the long term memories that were extracted to make a recommendation that the user will be sure to like.

现在，让我们开始一个新会话并询问晚餐做什么的建议。代理可以使用存储来访问被提取的长期记忆，以做出用户肯定会喜欢的推荐。

In [ ]:
config = {
    "configurable": {
        "thread_id": "session-2",  # New session ID
        "actor_id": actor_id,  # Same actor ID
    }
}

run_agent("Today's a new day, what should I make for dinner tonight?", config)

### Wrapping up

### 总结

As you can see, the agent received both pre-model hook context from the user preferences namespace search and was able to search on its own for long term memories in the fact namespace to create a comprehensive answer for the user.

如您所见，代理从用户偏好命名空间搜索中接收了模型前钩子上下文，并能够在事实命名空间中自行搜索长期记忆，为用户创建全面的答案。

The AgentCoreMemoryStore is very flexible and can be implemented in a variety of ways, including pre/post model hooks or just tools themselves with store operations. Used alongside the AgentCoreMemorySaver for checkpointing, both full conversational state and long term insights can be combined to form a complex and intelligent agent system.

AgentCoreMemoryStore 非常灵活，可以通过多种方式实现，包括模型前/后钩子或仅使用带有存储操作的工具本身。与用于检查点的 AgentCoreMemorySaver 一起使用，完整的对话状态和长期洞察都可以结合起来形成一个复杂而智能的代理系统。